<a href="https://colab.research.google.com/github/harry-8818/-From-Neural-Network-Foundations-to-Multi-Object-Tracking/blob/main/MLP_and_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# checking if the accelerator is availale if not we will use CPU
device = ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device used : {device}")

# converting the image to tensor data and scaling it down after converting to float
pipeline = v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)])

# loading the data for testing and training
training_data = datasets.FashionMNIST(root="image",train=True,download=True,transform=pipeline)
test_data = datasets.FashionMNIST(root="image",train=False,download=True,transform=pipeline)
train_dataloader = DataLoader(training_data,batch_size=100,shuffle=True)
test_dataloader = DataLoader(test_data,batch_size=100,shuffle=False)

# the blueprint (Multi-Layer Perceptron)
class build1_mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.initial_layer = nn.Flatten()
        self.hidden_and_output_layer = nn.Sequential(nn.Linear(28*28, 200),nn.ReLU(),
                                                     nn.Linear(200,100),nn.ReLU(),
                                                     nn.Linear(100,50),nn.ReLU(),
                                                     nn.Linear(50,10))
    # function taking the image as input and gives a final output tensor
    def forward(self,input_image):
        input_image = self.initial_layer(input_image)
        output = self.hidden_and_output_layer(input_image)
        return output

# initialising the network and shipping it to GPU
network = build1_mlp().to(device)
cost_fn = nn.CrossEntropyLoss()
# using Adam instead of SGD as it faster and more efficient
optimizer = torch.optim.Adam(network.parameters(),lr=1e-3)

# for training data
def train(dataloader,network,cost_fn,optimizer):
    size = len(dataloader.dataset)
    network.train()# essential to tell that we are in the learning phase now

    # forward pass
    for batch,(images,correct_labels) in enumerate(dataloader):
        images,correct_labels = images.to(device),correct_labels.to(device)
        predicted_output= network(images)
        cost = cost_fn(predicted_output,correct_labels)

        # backward pass (backpropagation)
        optimizer.zero_grad() # resets the previous gradients
        cost.backward()       # calculates the new gradients using chain rule
        optimizer.step()      # changes the weights and biases acc. to calculated gradients
        if batch % 100 == 0:
            cost,current = cost.item(),(batch + 1) * len(images)
            print(f"Cost: {cost:>7f}  [{current:>5d}/{size:>5d}]")

# for testing data
def test(dataloader,network,cost_fn):
    size = len(dataloader.dataset)
    batches = len(dataloader)
    network.eval() # turning on the evaluation mode -> stops training specific steps

    test_cost,correct = 0,0
    with torch.no_grad(): # since we are testing data we have to turn off the autograd engine
        for images,correct_labels in dataloader: # forward pass only
            images,correct_labels = images.to(device),correct_labels.to(device)
            predicted_output = network(images)
            test_cost += cost_fn(predicted_output,correct_labels).item()
            correct += (predicted_output.argmax(1) == correct_labels).type(torch.float).sum().item()

    test_cost /= batches
    correct /= size
    print(f"Test Results: \n Accuracy: {(100*correct):>0.1f}%, Average Cost: {test_cost:>8f} \n")
    return correct

# we will take epochs = 10 so that the network gets enough iterations to learn
# we are not using a number too large otherwise it will start memorizing the patterns of training data(overfitting)
import copy
epochs = 10
best_accuracy = 0.0
best_weights = None
for t in range(epochs):
    print(f"Executing Epoch Number {t+1} : \n")
    train(train_dataloader,network,cost_fn,optimizer)
    accuracy = test(test_dataloader,network,cost_fn)
    if accuracy > best_accuracy :
      best_accuracy = accuracy
      best_weights = copy.deepcopy(network.state_dict())
print(f"Highest accuracy across all epochs : {(100*best_accuracy):>0.1f}%")
# to avoid overfitting if it occuring to a slight extent too
network.load_state_dict(best_weights)

# testing the trained model
import os
from PIL import Image
import pandas as pd

TEST_DIR = "images"

@torch.no_grad()
def predict_folder(folder, network):
    network.eval()
    filenames = sorted(f for f in os.listdir(folder)if f.endswith(".png") and not f.startswith("._"))
    image_ids,predictions = [],[]
    for name in filenames:
        img = Image.open(os.path.join(folder,name))
        tensor = pipeline(img).unsqueeze(0).to(device)
        label = network(tensor).argmax(1).item()
        image_ids.append(name.replace(".png", ""))
        predictions.append(label)
    return image_ids,predictions

image_ids,predictions = predict_folder("images",network)

df = pd.DataFrame({"image_id":image_ids,"label":predictions})
df.to_csv("submission_mlp.csv", index=False)
print(df.shape)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# checking if the accelerator is availale if not we will use CPU
device = ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device used : {device}")

# converting the image to tensor data and scaling it down after converting to float
pipeline = v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32, scale=True)])

# loading the data for testing and training
training_data = datasets.FashionMNIST(root="image",train=True,download=True,transform=pipeline)
test_data = datasets.FashionMNIST(root="image",train=False,download=True,transform=pipeline)
train_dataloader = DataLoader(training_data,batch_size=100,shuffle=True)
test_dataloader = DataLoader(test_data,batch_size=100,shuffle=False)

# the blueprint
class build2_cnn(nn.Module):
    def __init__(self):
        super().__init__()
        # kernel and padding sizes are chosen such that no spatial dimensions change during iteration of filters
        # the spatial dimensions converge during the maxpool calls
        self.conv_layers = nn.Sequential(nn.Conv2d(1,32,kernel_size=3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                                         nn.Conv2d(32,64,kernel_size=3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.linear_layers = nn.Sequential(nn.Flatten(),nn.Linear(64*49,512),nn.ReLU(),nn.Linear(512, 10))

    def forward(self, input_image):
        final_conv_layer = self.conv_layers(input_image)
        output = self.linear_layers(final_conv_layer)
        return output

# initialising the network and shipping it to GPU
network = build2_cnn().to(device)
cost_fn = nn.CrossEntropyLoss()
# using Adam instead of SGD as it faster and more efficient
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)

# for training data
def train(dataloader,network,cost_fn,optimizer):
    size = len(dataloader.dataset)
    network.train()# essential to tell that we are in learning phase now

    # forward pass
    for batch, (images,correct_labels) in enumerate(dataloader):
        images,correct_labels = images.to(device),correct_labels.to(device)
        predicted_output= network(images)
        cost = cost_fn(predicted_output,correct_labels)

        # backward pass (backpropagation)
        optimizer.zero_grad() # resets the previous gradients
        cost.backward()       # calculates the new gradients using chain rule
        optimizer.step()      # changes the weights and biases acc. to calculated gradients
        if batch % 100 == 0:
            cost,current = cost.item(),(batch + 1) * len(images)
            print(f"Cost: {cost:>7f}  [{current:>5d}/{size:>5d}]")

# for testing data
def test(dataloader,network,cost_fn):
    size = len(dataloader.dataset)
    batches = len(dataloader)
    network.eval() # turning on the evaluation mode -> stops training specific steps

    test_cost,correct = 0,0
    with torch.no_grad(): # since we are testing data we have to turn off the autograd engine
        for images,correct_labels in dataloader: # forward pass only
            images,correct_labels = images.to(device),correct_labels.to(device)
            predicted_output = network(images)
            test_cost += cost_fn(predicted_output,correct_labels).item()
            correct += (predicted_output.argmax(1) == correct_labels).type(torch.float).sum().item()

    test_cost /= batches
    correct /= size
    print(f"Test Results: \n Accuracy: {(100*correct):>0.1f}%, Average Cost: {test_cost:>8f} \n")
    return correct

# we will take epochs = 10 so that the network gets enough iterations to learn
# we are not using a number too large otherwise it will start memorizing the patterns of training data(overfitting)
import copy
epochs = 10
best_accuracy = 0.0
best_weights = None
for t in range(epochs):
    print(f"Executing Epoch Number {t+1} : \n")
    train(train_dataloader,network,cost_fn,optimizer)
    accuracy = test(test_dataloader,network,cost_fn)
    if accuracy > best_accuracy :
      best_accuracy = accuracy
      best_weights = copy.deepcopy(network.state_dict())
print(f"Highest accuracy across all epochs : {(100*best_accuracy):>0.1f}%")
# to avoid overfitting if it occuring to a slight extent too
network.load_state_dict(best_weights)

# testing the trained model
import os
from PIL import Image
import pandas as pd

TEST_DIR = "images"

@torch.no_grad()
def predict_folder(folder, network):
    network.eval()
    filenames = sorted(f for f in os.listdir(folder)if f.endswith(".png") and not f.startswith("._"))
    image_ids,predictions = [],[]
    for name in filenames:
        img = Image.open(os.path.join(folder,name))
        tensor = pipeline(img).unsqueeze(0).to(device)
        label = network(tensor).argmax(1).item()
        image_ids.append(name.replace(".png", ""))
        predictions.append(label)
    return image_ids,predictions

image_ids,predictions = predict_folder("images",network)

df = pd.DataFrame({"image_id":image_ids,"label":predictions})
df.to_csv("submission_cnn.csv", index=False)
print(df.shape)